In [ ]:
%pip install rasterio
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import h5py
import numpy as np
from PIL import Image
from scipy.ndimage import map_coordinates


# ===========================================================================
# CONFIG — edit these two lines
# ===========================================================================

DATA_DIR   = "/Users/maddiekimmel/Desktop/IcyCape"
OUTPUT_DIR = "/Users/maddiekimmel/Documents/GeoTiff_IcyCape"

# ===========================================================================
# OUTPUT GEOTIFF FORMAT
# -  CRS:    EPSG:32604  (WGS 84 / UTM Zone 4N, metres)
# -  Band 1: sigma0 backscatter (float32, linear power)
# -  Band 2: grayscale PNG render (float32, 0.0 – 1.0)
#
# GEOREFERENCING stored as proper double-type GeoTIFF tags:
#   33550  ModelPixelScaleTag   — pixel size in metres (x, y)
#   33922  ModelTiepointTag     — top-left pixel tied to UTM coordinate
#   34736  GeoKeyDirectoryTag   — EPSG:32604 declared as projected CRS
#   34737  GeoAsciiParamsTag    — CRS name string "WGS 84 / UTM zone 4N"
#
# Reading georeferencing back in Python:
#   with tifffile.TiffFile(path) as tif:
#       tags     = {t.code: t for t in tif.pages[0].tags.values()}
#       scale    = tags[33550].value   # (x_scale, y_scale, 0.0) in metres
#       tp       = tags[33922].value   # (0,0,0, x_min, y_max, 0)
#       epsg     = tags[34736].value[15]  # 32604
#       x_min    = tp[3]
#       y_max    = tp[4]
#       x_max    = x_min + scale[0] * data.shape[-1]
#       y_min    = y_max - scale[1] * data.shape[-2]
# ===========================================================================


# ---------------------------------------------------------------------------
# UTM Zone 4N projection  —  EPSG:32604  WGS84 / UTM Zone 4N
# ---------------------------------------------------------------------------

def utm_fwd(lon_deg, lat_deg, zone=4):
    """WGS84 lon/lat (degrees) -> UTM Zone 4N easting/northing (metres)."""
    a   = 6378137.0
    f   = 1 / 298.257223563
    e2  = 2*f - f**2
    ep2 = e2 / (1 - e2)
    k0  = 0.9996
    E0  = 500000.0
    lam0 = np.radians(-180 + (zone - 1)*6 + 3)
    phi = np.radians(lat_deg)
    lam = np.radians(lon_deg)
    N  = a / np.sqrt(1 - e2 * np.sin(phi)**2)
    T  = np.tan(phi)**2
    C  = ep2 * np.cos(phi)**2
    A_ = np.cos(phi) * (lam - lam0)
    e4 = e2**2;  e6 = e2**3
    M  = a * ((1 - e2/4 - 3*e4/64 - 5*e6/256)   * phi
              - (3*e2/8 + 3*e4/32 + 45*e6/1024)  * np.sin(2*phi)
              + (15*e4/256 + 45*e6/1024)          * np.sin(4*phi)
              - (35*e6/3072)                       * np.sin(6*phi))
    x = k0*N*(A_ + (1 - T + C)*A_**3/6
              + (5 - 18*T + T**2 + 72*C - 58*ep2)*A_**5/120) + E0
    y = k0*(M + N*np.tan(phi)*(A_**2/2
              + (5 - T + 9*C + 4*C**2)*A_**4/24
              + (61 - 58*T + T**2 + 600*C - 330*ep2)*A_**6/720))
    return x, y


def utm_inv(x, y, zone=4):
    """UTM Zone 4N easting/northing (metres) -> WGS84 lon/lat (degrees)."""
    a   = 6378137.0
    f   = 1 / 298.257223563
    e2  = 2*f - f**2
    ep2 = e2 / (1 - e2)
    k0  = 0.9996
    E0  = 500000.0
    lam0 = np.radians(-180 + (zone - 1)*6 + 3)
    e1  = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
    M   = y / k0
    mu  = M / (a*(1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))
    phi1 = (mu
            + (3*e1/2   - 27*e1**3/32)    * np.sin(2*mu)
            + (21*e1**2/16 - 55*e1**4/32) * np.sin(4*mu)
            + (151*e1**3/96)               * np.sin(6*mu)
            + (1097*e1**4/512)             * np.sin(8*mu))
    N1 = a / np.sqrt(1 - e2*np.sin(phi1)**2)
    T1 = np.tan(phi1)**2
    C1 = ep2 * np.cos(phi1)**2
    R1 = a*(1 - e2) / (1 - e2*np.sin(phi1)**2)**1.5
    D  = (x - E0) / (N1*k0)
    phi = phi1 - (N1*np.tan(phi1)/R1)*(
          D**2/2
        - (5 + 3*T1 + 10*C1 - 4*C1**2 - 9*ep2)*D**4/24
        + (61 + 90*T1 + 298*C1 + 45*T1**2 - 252*ep2 - 3*C1**2)*D**6/720)
    lam = lam0 + (D
        - (1 + 2*T1 + C1)*D**3/6
        + (5 - 2*C1 + 28*T1 - 3*C1**2 + 8*ep2 + 24*T1**2)*D**5/120
    ) / np.cos(phi1)
    return np.degrees(lam), np.degrees(phi)


# ---------------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------------

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

stems = []
for kml in sorted(Path(DATA_DIR).glob("*.kml")):
    stem = kml.stem
    if (Path(DATA_DIR) / f"{stem}.mat").exists() and \
       (Path(DATA_DIR) / f"{stem}.png").exists():
        stems.append(stem)
    else:
        print(f"[SKIP] {stem} — missing .mat or .png")

print(f"Found {len(stems)} scene(s), converting to GeoTIFF (EPSG:32604)...")
print()

for stem in stems:
    try:

        # ── Read KML bounds ────────────────────────────────────────────────
        tree  = ET.parse(Path(DATA_DIR) / f"{stem}.kml")
        root  = tree.getroot()
        ns    = re.match(r"\{.*\}", root.tag)
        ns    = ns.group(0) if ns else ""
        box   = root.find(f".//{ns}LatLonBox")
        north = float(box.find(f"{ns}north").text.strip())
        south = float(box.find(f"{ns}south").text.strip())
        east  = float(box.find(f"{ns}east").text.strip())
        west  = float(box.find(f"{ns}west").text.strip())

        # ── Read PNG ───────────────────────────────────────────────────────
        png = np.array(
            Image.open(Path(DATA_DIR) / f"{stem}.png").convert("L"),
            dtype=np.uint8
        )

        # ── Read MAT (MATLAB v7.3 = HDF5) ─────────────────────────────────
        arrays = {}
        with h5py.File(Path(DATA_DIR) / f"{stem}.mat", "r") as f:
            def _load(name, obj):
                if isinstance(obj, h5py.Dataset):
                    arr = obj[()]
                    if arr.ndim >= 2:
                        arr = arr.T    # MATLAB column-major -> row-major
                    arrays[name.lstrip("/")] = arr
            f.visititems(_load)

        sigma0 = arrays["A_grid"].astype(np.float32)
        lat    = arrays["LAT_grid"].astype(np.float64)

        # ── Align shapes ───────────────────────────────────────────────────
        h_src, w_src = lat.shape
        sigma0 = sigma0[:h_src, :w_src]
        if png.shape != (h_src, w_src):
            png = np.array(
                Image.fromarray(png).resize((w_src, h_src), Image.BILINEAR)
            )

        # Flip .mat to north-up (row 0 = north) to match PNG orientation
        sigma0 = np.flipud(sigma0)
        sigma0 = np.where(sigma0 < 0, 0.0, sigma0)

        # ── Compute projected bounding box ─────────────────────────────────
        corners_lon = np.array([west, east, west, east])
        corners_lat = np.array([north, north, south, south])
        cx, cy      = utm_fwd(corners_lon, corners_lat)

        x_min, x_max = cx.min(), cx.max()
        y_min, y_max = cy.min(), cy.max()

        out_w = w_src
        out_h = h_src
        pixel_size_x = (x_max - x_min) / out_w
        pixel_size_y = (y_max - y_min) / out_h

        # ── Build output grid in UTM space ─────────────────────────────────
        xs = np.linspace(x_min, x_max, out_w)
        ys = np.linspace(y_max, y_min, out_h)
        XX, YY = np.meshgrid(xs, ys)

        # ── Inverse-project each output pixel to source lat/lon ────────────
        src_lon, src_lat = utm_inv(XX, YY)

        src_row = (north - src_lat) / (north - south) * (h_src - 1)
        src_col = (src_lon - west)  / (east  - west)  * (w_src - 1)

        outside = (
            (src_row < 0) | (src_row > h_src - 1) |
            (src_col < 0) | (src_col > w_src - 1)
        )

        # ── Resample both bands into the UTM grid ──────────────────────────
        sigma0_proj = map_coordinates(
            sigma0.astype(np.float64), [src_row, src_col],
            order=1, mode="constant", cval=np.nan
        ).astype(np.float32)
        sigma0_proj[outside] = np.nan

        png_proj = map_coordinates(
            png.astype(np.float64), [src_row, src_col],
            order=1, mode="constant", cval=0
        ).astype(np.float32) / 255.0
        png_proj[outside] = 0.0

        # ── Write GeoTIFF with rasterio ───────────────────────────────────
        # rasterio writes proper georeferencing so you can load it back with:
        #   import rasterio
        #   with rasterio.open(path) as src:
        #       print(src.crs)      # EPSG:32604
        #       print(src.bounds)   # left, bottom, right, top in metres
        #       sigma0 = src.read(1)
        #       png    = src.read(2)
        import rasterio
        from rasterio.transform import from_bounds
        from rasterio.crs import CRS

        out_path  = Path(OUTPUT_DIR) / f"{stem}.tif"
        transform = from_bounds(x_min, y_min, x_max, y_max, out_w, out_h)

        with rasterio.open(
            str(out_path),
            mode        = "w",
            driver      = "GTiff",
            height      = out_h,
            width       = out_w,
            count       = 2,
            dtype       = np.float32,
            crs         = CRS.from_epsg(32604),
            transform   = transform,
            compress    = "deflate",
        ) as dst:
            dst.write(sigma0_proj, 1)   # band 1 = sigma0
            dst.write(png_proj,    2)   # band 2 = grayscale PNG
            dst.update_tags(
                band1 = "sigma0_linear_power",
                band2 = "grayscale_png_0_to_1",
                crs   = "EPSG:32604",
            )

        size_mb = out_path.stat().st_size / 1024 / 1024
        print(f"[OK] {stem}.tif  ({size_mb:.1f} MB)")

    except Exception as e:
        print(f"[FAIL] {stem}: {e}")

print()
print(f"Done. GeoTIFFs saved to: {Path(OUTPUT_DIR).resolve()}")
print()
print("To load back in Python:")
print("  import rasterio")
print("  with rasterio.open(path) as src:")
print("      print(src.crs)      # EPSG:32604")
print("      print(src.bounds)   # left, bottom, right, top in metres")
print("      sigma0 = src.read(1)")
print("      png    = src.read(2)")